# Kiểm tra MinerU + RAG-Anything (AMI service) trên Colab

Notebook đo **VRAM / thời gian của MinerU** và chạy thử luồng RAG thật với đúng bộ thư viện của service
`ami_rag` (`raganything` fork + `mineru[core]<4` + `lightrag-hku<1.5`). Không cần Mongo/Qdrant/Redis/MinIO/API key:
LightRAG dùng storage local, LLM/embedding là bản giả.

**Cách dùng:** `Runtime ▸ Change runtime type ▸ GPU (T4)` rồi chạy lần lượt từ trên xuống.
Để so sánh CPU: chạy lại với runtime *CPU* (phần GPU sẽ tự bỏ qua).

**Lưu ý về con số VRAM**: đo bằng `nvidia-smi` (tổng `memory.used` cao nhất trừ baseline), tức gồm CUDA context
và bộ nhớ PyTorch cache giữ lại — đây là mức cần để *dimension GPU*, cao hơn `torch.cuda.max_memory_allocated`.
Colab T4 có 15 GB; production dùng GPU khác nên chỉ nên đọc số tuyệt đối, không đọc tốc độ.

**Phát hiện cần biết:** MinerU 4.x đổi CLI (`mineru parse`), fork gọi CLI 3.x (`mineru -p ... -o ...`) nên phải ghim `mineru<4`.
MinerU 3.4.x mặc định backend `hybrid-engine` (nặng VRAM) nếu không truyền `-b`; CLI 3.x chọn thiết bị bằng biến môi trường
`MINERU_DEVICE_MODE` (không có cờ `-d`).

In [ ]:
# 0. Môi trường
import subprocess, shutil, sys
print(sys.version)
print(subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True).stdout or "KHÔNG có GPU (runtime CPU)")
print(subprocess.run("free -g | head -2; df -h /content | tail -1; nproc", shell=True, capture_output=True, text=True).stdout)
HAS_GPU = shutil.which("nvidia-smi") is not None and subprocess.run("nvidia-smi -L", shell=True, capture_output=True).returncode == 0
print("HAS_GPU =", HAS_GPU)

In [ ]:
# 1. Cấu hình
GIT_URL = "https://github.com/P-Bao/RAG-Anything.git"
GIT_BRANCH = "feature/ami-service"   # nhánh phải đã được push lên origin

MAX_PAGES = 10            # số trang PDF dùng để đo (0-based: trang 0..MAX_PAGES-1)
CPU_PAGES = 3             # số trang cho lần chạy CPU (chậm)
LANG = None               # ví dụ "ch", "en"... để None = mặc định MinerU. Cần thử mã cho tiếng Việt.
VRAM_SWEEP = [4, 8, 16]   # giả lập VRAM (GB) qua MINERU_VIRTUAL_VRAM_SIZE: MinerU chọn batch size theo giá trị này
RUN_HYBRID = True         # đo thêm backend mặc định hybrid-engine để so sánh
RUN_RERANKER = True       # đo VRAM bge-reranker-v2-m3
MODEL_SOURCE = "huggingface"   # hoặc "modelscope"

In [ ]:
# 2. Lấy mã nguồn RAG-Anything (fork + ami_rag) bằng git clone
import os
from pathlib import Path

REPO = Path("/content/RAG-Anything")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-B", GIT_BRANCH, "FETCH_HEAD"], check=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_URL, str(REPO)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone lỗi (nhánh {GIT_BRANCH} đã được push chưa?):\n{r.stderr}")
assert (REPO / "ami_rag").is_dir() and (REPO / "raganything").is_dir(), "Thiếu ami_rag/ hoặc raganything/"
print(subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-3"], capture_output=True, text=True).stdout)

In [ ]:
# 3. Cài thư viện giống service (mineru[core]<4, lightrag-hku<1.5 lấy từ pyproject của fork)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{REPO}[service]", "pytest", "pytest-asyncio", "pandas"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-3000:])
import importlib.metadata as md
for pkg in ("mineru", "lightrag-hku", "torch", "transformers", "raganything"):
    try:
        print(f"{pkg:14s}", md.version(pkg))
    except Exception as e:
        print(f"{pkg:14s} KHÔNG cài được: {e}")
assert int(md.version("mineru").split(".")[0]) < 4, "mineru phải < 4 (CLI khác với fork)"
print(subprocess.run(["mineru", "--version"], capture_output=True, text=True).stdout)

In [ ]:
# 4. PDF mẫu: upload file của bạn (khuyến nghị: PDF tiếng Việt có bảng + ảnh) hoặc dùng bài báo mẫu
import urllib.request
PDF = Path("/content/sample.pdf")
USE_UPLOAD = True
if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    if up:
        PDF.write_bytes(next(iter(up.values())))
if not PDF.exists():
    urllib.request.urlretrieve("https://arxiv.org/pdf/1706.03762", PDF)   # Attention Is All You Need (15 trang, có bảng/hình)
import pypdfium2 as pdfium
n_pages = len(pdfium.PdfDocument(str(PDF)))
print(PDF, f"{PDF.stat().st_size/1e6:.2f} MB", n_pages, "trang")
MAX_PAGES = min(MAX_PAGES, n_pages)

In [ ]:
# 5. Công cụ đo: VramMonitor + run_mineru
import json, time, threading
import pandas as pd

def gpu_used_mib():
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
        return sum(int(x) for x in out.split())
    except Exception:
        return 0

class VramMonitor:
    """Lấy mẫu tổng GPU memory.used; peak_delta_mib = đỉnh - baseline lúc bắt đầu."""
    def __init__(self, interval=0.25):
        self.interval = interval
    def __enter__(self):
        self.baseline = self.peak = gpu_used_mib()
        self._stop = threading.Event()
        def loop():
            while not self._stop.is_set():
                self.peak = max(self.peak, gpu_used_mib())
                time.sleep(self.interval)
        self._t = threading.Thread(target=loop, daemon=True)
        self._t.start()
        return self
    def __exit__(self, *exc):
        self._stop.set()
        self._t.join()
        self.peak = max(self.peak, gpu_used_mib())
    @property
    def peak_delta_mib(self):
        return self.peak - self.baseline

def mineru_env(device=None, virtual_vram_gb=None):
    env = dict(os.environ)
    env["MINERU_MODEL_SOURCE"] = MODEL_SOURCE
    for k in ("MINERU_DEVICE_MODE", "MINERU_VIRTUAL_VRAM_SIZE"):
        env.pop(k, None)
    if device:
        env["MINERU_DEVICE_MODE"] = device
    if virtual_vram_gb:
        env["MINERU_VIRTUAL_VRAM_SIZE"] = str(virtual_vram_gb)
    return env

def run_mineru(label, backend="pipeline", device=None, virtual_vram_gb=None, pages=None, timeout=3600):
    pages = pages or MAX_PAGES
    out_dir = Path("/content/mineru_out") / label
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True)
    cmd = ["mineru", "-p", str(PDF), "-o", str(out_dir), "-m", "auto", "-b", backend,
           "-s", "0", "-e", str(pages - 1)]
    if LANG:
        cmd += ["-l", LANG]
    t0 = time.time()
    with VramMonitor() as mon:
        p = subprocess.run(cmd, env=mineru_env(device, virtual_vram_gb), capture_output=True, text=True, timeout=timeout)
    row = dict(label=label, backend=backend, device=device or "auto", virtual_vram_gb=virtual_vram_gb,
               pages=pages, seconds=round(time.time() - t0, 1), peak_vram_mib=mon.peak_delta_mib,
               ok=p.returncode == 0, out_dir=str(out_dir))
    if p.returncode != 0:
        row["error"] = (p.stderr or p.stdout)[-800:]
    return row

results = []
def record(row):
    results.append(row)
    print({k: v for k, v in row.items() if k not in ("out_dir",)})
    return row

In [ ]:
# 6. Warm-up (tải model lần đầu, KHÔNG tính vào kết quả)
if HAS_GPU:
    w = run_mineru("warmup", "pipeline", device="cuda", pages=1)
    print("warm-up:", {k: v for k, v in w.items() if k != "out_dir"})

In [ ]:
# 7. pipeline trên GPU với các mức VRAM giả lập (VRAM_SWEEP) và mức tự nhận
if HAS_GPU:
    for gb in [None] + VRAM_SWEEP:
        record(run_mineru(f"pipeline-gpu-vram{gb or 'auto'}", "pipeline", device="cuda", virtual_vram_gb=gb))
    if RUN_HYBRID:
        record(run_mineru("hybrid-engine-gpu-default", "hybrid-engine", device="cuda"))
else:
    print("Bỏ qua: không có GPU")

In [ ]:
# 8. pipeline trên CPU (chậm: dùng ít trang)
record(run_mineru("pipeline-cpu", "pipeline", device="cpu", pages=min(CPU_PAGES, MAX_PAGES)))

In [ ]:
# 9. Bảng kết quả + chất lượng đầu ra (số item theo loại, ảnh)
df = pd.DataFrame(results)
display(df.drop(columns=["out_dir", "error"], errors="ignore"))

def summarize_output(out_dir):
    out_dir = Path(out_dir)
    cl = next(iter(out_dir.rglob("*_content_list.json")), None)
    if not cl:
        return {"content_list": None}
    items = json.loads(cl.read_text())
    from collections import Counter
    return {"content_list": cl.name, "items": dict(Counter(i.get("type") for i in items)),
            "images_dir_files": sum(1 for _ in cl.parent.rglob("images/*"))}

for r in results:
    if r["ok"]:
        print(r["label"], summarize_output(r["out_dir"]))

In [ ]:
# 10. Luồng thư viện thật: RAGAnything.parse_document (MinerU) -> insert_content_list -> aquery_data
#     LightRAG local + LLM/embedding giả; MinerU chạy bằng đúng tham số worker sẽ dùng.
import numpy as np, uuid
from lightrag import LightRAG
from lightrag.utils import EmbeddingFunc
from raganything import RAGAnything, RAGAnythingConfig

async def fake_llm(prompt, system_prompt=None, history_messages=None, **kw):
    if "detailed_description" in f"{system_prompt or ''}\n{prompt}":
        return json.dumps({"detailed_description": "Mô tả giả.",
                           "entity_info": {"entity_name": "Media", "entity_type": "media", "summary": "Tóm tắt giả."}},
                          ensure_ascii=False)
    return "entity<|#|>Demo<|#|>Organization<|#|>demo.\n<|COMPLETE|>"

async def fake_vision(prompt, system_prompt=None, history_messages=None, image_data=None, **kw):
    return await fake_llm(prompt, system_prompt, history_messages)

async def fake_embed(texts):
    return np.random.default_rng(0).random((len(texts), 8), dtype=np.float32)

WORK = Path("/content/rag_work"); shutil.rmtree(WORK, ignore_errors=True); WORK.mkdir()
rag = LightRAG(working_dir=str(WORK / "rag_storage"), workspace="t" + uuid.uuid4().hex[:6],
               kv_storage="JsonKVStorage", vector_storage="NanoVectorDBStorage",
               graph_storage="NetworkXStorage", doc_status_storage="JsonDocStatusStorage",
               llm_model_func=fake_llm, embedding_func=EmbeddingFunc(embedding_dim=8, func=fake_embed))
await rag.initialize_storages()
anything = RAGAnything(lightrag=rag, vision_model_func=fake_vision,
                       config=RAGAnythingConfig(working_dir=str(WORK / "rag_storage"), parser="mineru", parse_method="auto"))

os.environ["MINERU_MODEL_SOURCE"] = MODEL_SOURCE
os.environ.pop("MINERU_VIRTUAL_VRAM_SIZE", None)
if HAS_GPU:
    os.environ["MINERU_DEVICE_MODE"] = "cuda"
else:
    os.environ["MINERU_DEVICE_MODE"] = "cpu"

kwargs = dict(backend="pipeline", start_page=0, end_page=(MAX_PAGES if HAS_GPU else min(CPU_PAGES, MAX_PAGES)) - 1)
if LANG:
    kwargs["lang"] = LANG
t0 = time.time()
with VramMonitor() as mon:
    content_list, _ = await anything.parse_document(str(PDF), output_dir=str(WORK / "parsed"), parse_method="auto", **kwargs)
parse_s = time.time() - t0
from collections import Counter
print(f"parse_document: {parse_s:.1f}s, peak VRAM {mon.peak_delta_mib} MiB, items={dict(Counter(i.get('type') for i in content_list))}")
results.append(dict(label="library-parse_document", backend="pipeline", device="cuda" if HAS_GPU else "cpu",
                    seconds=round(parse_s, 1), peak_vram_mib=mon.peak_delta_mib, ok=True))

await anything.insert_content_list(content_list, file_path=PDF.name, doc_id="demo-doc")
found = await anything.aquery_data("table results", mode="naive", chunk_top_k=30)
chunks = found["data"]["chunks"]
print("modality:", dict(Counter(c["modality"] for c in chunks)))
for c in chunks:
    if c["modality"] == "table":
        print("TABLE page", c["page_idx"], "|", (c["table_body"] or "")[:200].replace("\n", " "))
        break
await rag.finalize_storages()

In [ ]:
# 11. VRAM của reranker bge-reranker-v2-m3 (fp16, 40 cặp x 512 token) — chạy cuối để không lẫn baseline
if RUN_RERANKER and HAS_GPU:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    name = "BAAI/bge-reranker-v2-m3"
    base = gpu_used_mib()
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name, torch_dtype=torch.float16).cuda().eval()
    pairs = [["học phí năm 2026 là bao nhiêu?", ("Đoạn văn mẫu về tuyển sinh và học phí. " * 60)] for _ in range(40)]
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    with torch.no_grad():
        enc = tok(pairs, padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
        scores = model(**enc).logits
        torch.cuda.synchronize()
    rr = dict(label="reranker-fp16-40x512", seconds=round(time.time() - t0, 2),
              peak_vram_mib=gpu_used_mib() - base, torch_allocated_mib=round(torch.cuda.max_memory_allocated() / 2**20),
              torch_reserved_mib=round(torch.cuda.max_memory_reserved() / 2**20), ok=True)
    results.append(rr)
    print(rr)
else:
    print("Bỏ qua reranker")

In [ ]:
# 12. Tổng hợp + khuyến nghị cấu hình .env
df = pd.DataFrame(results)
display(df.drop(columns=["out_dir", "error"], errors="ignore"))

gpu_pipe = df[(df.get("backend") == "pipeline") & (df.get("device") != "cpu") & (df["ok"])] if len(df) else df
if HAS_GPU and len(gpu_pipe):
    worst = int(gpu_pipe["peak_vram_mib"].max())
    auto = df[df["label"] == "pipeline-gpu-vramauto"]
    print(f"\nĐỉnh VRAM pipeline (mọi cấu hình đã đo): {worst} MiB (~{worst/1024:.1f} GiB)")
    if len(auto):
        print(f"Mức tự nhận (batch lớn nhất theo VRAM thật của GPU này): {int(auto.peak_vram_mib.iloc[0])} MiB")
    print("=> Chọn GPU với VRAM >= đỉnh + ~1-2 GiB dự phòng (fragmentation/PDF dài hơn mẫu).")
    print("=> Nếu GPU chung với service khác: đặt MINERU_VIRTUAL_VRAM_SIZE = mức VRAM bạn muốn MinerU 'nghĩ' mình có")
    print("   (giá trị nhỏ -> batch nhỏ -> ít VRAM hơn, chậm hơn) và luôn truyền backend=pipeline.")
print("\nKhuyến nghị .env cho ami_rag (sau khi thêm biến vào settings):")
print("  MINERU_BACKEND=pipeline   # tránh mặc định hybrid-engine của MinerU 3.4.x")
print("  MINERU_DEVICE=cuda        # hoặc cpu / để trống = tự nhận; truyền qua MINERU_DEVICE_MODE")
print("  MINERU_VIRTUAL_VRAM_SIZE=<GB>   # tuỳ chọn, giới hạn batch size theo VRAM thực dùng")
df.to_csv("/content/mineru_vram_results.csv", index=False)
print("Đã lưu /content/mineru_vram_results.csv")

## Gửi lại kết quả
Dán bảng ở cell 12 (hoặc tải `/content/mineru_vram_results.csv`) để quyết định: GPU tối thiểu, `MINERU_VIRTUAL_VRAM_SIZE`,
và có thể dùng GPU chung với service khác hay không.